- Model serving frameworks: vLLM, TGI, TensorRT-LLM, Triton Inference Server
- PagedAttention: how vLLM manages KV-cache memory
- Batching strategies: static, dynamic, continuous
- Prefill vs decode phases and their different compute profiles
- Autoscaling: GPU utilization metrics, request queuing
- Streaming responses: SSE, WebSocket implementations
- Multi-model serving and model routing
- Cost optimization: spot instances, right-sizing GPUs

# **Appendix: vLLM — What Production LLM Serving Actually Looks Like**

![](https://vllm.ai/blog-assets/figures/2025-vllm-anatomy/engine_constructor.png)
---

In this notebook we built generation from scratch — KV-caching, batching, continuous batching, speculative decoding. In production, engineers don't rewrite any of this. They use **vLLM**.

vLLM is an open-source LLM serving engine that packages all these techniques (and more) into a single system. You load a Hugging Face model, start a server, and get an OpenAI-compatible API endpoint. Under the hood, it handles everything automatically.

---

**The core innovation: PagedAttention**

We saw in Section 4 that the KV-cache grows with every token. In a real server handling hundreds of concurrent requests, KV-cache memory becomes the bottleneck.

The naive approach allocates a fixed block of GPU memory per request based on the maximum possible sequence length. If you reserve space for 2048 tokens but the request only uses 200, the rest is wasted. Traditional systems waste 60–80% of KV-cache memory this way.

PagedAttention borrows an idea from operating systems. Instead of one big contiguous block per request, the KV-cache is split into small fixed-size **blocks** (e.g., 16 tokens each) that can live anywhere in GPU memory. A block table maps each request's logical positions to physical blocks. New blocks are allocated on demand as the sequence grows, and freed immediately when a request finishes.

The result: memory waste drops from 60–80% to under 4%. The same GPU can serve 2–4x more concurrent requests.

---

**What vLLM handles for you**

Everything we built manually, plus much more:

| Feature | What it does | Our notebook equivalent |
|---------|-------------|----------------------|
| **PagedAttention** | Efficient KV-cache memory via paging | We used naive contiguous allocation |
| **Continuous batching** | Swaps finished/new requests every step | Section 7 |
| **Speculative decoding** | Draft + verify with EAGLE/draft models | Section 8 |
| **Chunked prefill** | Splits long prompts into chunks to avoid blocking decode | Not covered |
| **Prefix caching** | Shares KV-cache across requests with the same system prompt | Not covered |
| **Quantization** | FP8, INT4, INT8, AWQ, GPTQ | Not covered |
| **Tensor parallelism** | Splits one model across multiple GPUs | Not covered |
| **CUDA graphs** | Pre-records GPU ops to cut kernel launch overhead | Not possible in raw PyTorch loops |
| **OpenAI-compatible API** | Drop-in replacement for OpenAI endpoints | We had no API layer |

---

**Using vLLM in practice**

Starting a server:

```bash
pip install vllm

vllm serve meta-llama/Llama-3.1-8B-Instruct
```

This gives you an OpenAI-compatible endpoint at `http://localhost:8000`:

```python
from openai import OpenAI

client = OpenAI(base_url="http://localhost:8000/v1", api_key="unused")

response = client.chat.completions.create(
    model="meta-llama/Llama-3.1-8B-Instruct",
    messages=[{"role": "user", "content": "Explain KV-caching in one sentence."}],
)
print(response.choices[0].message.content)
```

Adding speculative decoding is one extra flag:

```bash
vllm serve meta-llama/Llama-3.1-70B-Instruct \
    --speculative-model meta-llama/Llama-3.2-1B-Instruct \
    --num-speculative-tokens 5
```

Adding quantization:

```bash
vllm serve meta-llama/Llama-3.1-70B-Instruct \
    --quantization fp8
```

No custom generation loops, no manual KV-cache management, no batch scheduling code.

---

**Why we still wrote everything from scratch**

Understanding how these systems work matters because you can debug performance issues when vLLM isn't behaving as expected, make informed decisions about which features to enable, reason about costs by understanding memory-bound vs compute-bound tradeoffs, and know what to measure (acceptance rate, time-to-first-token, inter-token latency, throughput).

The concepts in this notebook are the foundation. vLLM is how you deploy them.